In [1]:
%load_ext autoreload
%autoreload 2 

In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

In [ ]:
results_dir = Path("/home/gquetel/experiences-results/2026-02-16-results/")

datasets = ["AdventureWorks", "OHR", "OurAirports", "sakila"]

dataset_letters = {
    "OurAirports": "A",
    "sakila": "B",
    "AdventureWorks": "C",
    "OHR": "D",
}
all_letters = set(dataset_letters.values())


MODELS = [
    {"prefix": "ae_li",         "label": "Li + AE",         "results_dir": results_dir},
    {"prefix": "ae_securebert", "label": "SecureBERT + AE",  "results_dir": results_dir},
    {"prefix": "ae_gaur",       "label": "GAUR + AE",        "results_dir": results_dir},
]


def leave_one_out_complement(letter):
    """Return sorted complement letters for leave-one-out training."""
    return "".join(sorted(all_letters - {letter}))


def load_results(results_dir, model_prefix):
    """Load results CSVs using the new directory structure.

    Generic path:     {results_dir}/{model_prefix}_generic/{model_prefix}_{complement}_on_{letter}/results.csv
    Specialised path: {results_dir}/{model_prefix}_specialised/{model_prefix}_{letter}_on_{letter}/results.csv
    """
    prefix = model_prefix
    results = []
    for dataset in datasets:
        letter = dataset_letters[dataset]
        complement = leave_one_out_complement(letter)
        for model_type in ["generic", "specialised"]:
            if model_type == "generic":
                subdir = f"{prefix}_generic/{model_prefix}_{complement}_on_{letter}"
            else:
                subdir = f"{prefix}_specialised/{model_prefix}_{letter}_on_{letter}"
            path = results_dir / subdir / "results.csv"
            if path.exists():
                df = pd.read_csv(path)
                df["dataset"] = dataset
                df["type"] = model_type
                results.append(df)
    if not results:
        return pd.DataFrame()
    df = pd.concat(results, ignore_index=True)
    pct_cols = ["fone", "accuracy", "precision", "recall", "fpr",
                "balanced_accuracy_per_technique"]
    for col in pct_cols:
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].str.rstrip("%").astype(float)
    recall_cols = [c for c in df.columns if c.startswith("recall") and c != "recall"]
    for col in recall_cols:
        if df[col].dtype == object:
            df[col] = df[col].str.rstrip("%").astype(float)
    return df


# Load results for all registered models
all_results = {m["prefix"]: load_results(m["results_dir"], m["prefix"]) for m in MODELS}
for m in MODELS:
    print(f"{m['prefix']}: {len(all_results[m['prefix']])} rows")

## Main Metrics Comparison

In [ ]:
main_metrics = ["fone", "rocauc", "auprc", "balanced_accuracy_per_technique"]

metric_labels = {
    "accuracy": "Accuracy (%)",
    "precision": "Precision (%)",
    "recall": "Recall (%)",
    "fone": "F1 Score (%)",
    "rocauc": "AUROC",
    "auprc": "AUPRC",
    "balanced_accuracy_per_technique": "Balanced Accuracy (%)",
}

colors = {"generic": "#636EFA", "specialised": "#EF553B"}

def plot_main_metrics(results_df, title):
    if results_df.empty:
        print(f"No data available for: {title}")
        return None
    fig = make_subplots(
        rows=1, cols=len(main_metrics),
        subplot_titles=[metric_labels[m] for m in main_metrics],
        vertical_spacing=0.15
    )
    for idx, metric in enumerate(main_metrics):
        col = idx + 1
        for model_type in ["generic", "specialised"]:
            subset = results_df[results_df["type"] == model_type]
            fig.add_trace(
                go.Bar(
                    x=subset["dataset"],
                    y=subset[metric],
                    name=model_type.capitalize(),
                    marker_color=colors[model_type],
                    showlegend=(idx == 0),
                    legendgroup=model_type
                ),
                row=1, col=col
            )
    fig.update_layout(
        title=title,
        barmode="group",
        height=300,
        width=1300
    )
    fig.show()
    return fig

figs_metrics = {}
for m in MODELS:
    figs_metrics[m["prefix"]] = plot_main_metrics(
        all_results[m["prefix"]], f"Generic vs Specialised ({m['label']})"
    )

## ROC Curves Comparison

In [ ]:
def plot_roc_curves(results_df, results_dir, model_prefix, title):
    if results_df.empty:
        print(f"No data available for: {title}")
        return None
    prefix = model_prefix
    fig = make_subplots(
        rows=2, cols=2, subplot_titles=datasets, vertical_spacing=0.12
    )
    for idx, dataset in enumerate(datasets):
        row = idx // 2 + 1
        col = idx % 2 + 1
        letter = dataset_letters[dataset]
        complement = leave_one_out_complement(letter)
        for model_type in ["generic", "specialised"]:
            if model_type == "generic":
                subdir = f"{prefix}_generic/{model_prefix}_{complement}_on_{letter}"
                roc_filename = f"{model_prefix}_{complement}.csv"
            else:
                subdir = f"{prefix}_specialised/{model_prefix}_{letter}_on_{letter}"
                roc_filename = f"{model_prefix}_{letter}.csv"
            roc_dir = results_dir / subdir / "roc_curves"
            roc_path = roc_dir / roc_filename
            # Fallback: first CSV in roc_curves/ (handles non-standard naming)
            if not roc_path.exists() and roc_dir.exists():
                candidates = list(roc_dir.glob("*.csv"))
                if candidates:
                    roc_path = candidates[0]
            if roc_path.exists():
                roc_df = pd.read_csv(roc_path)
                fig.add_trace(
                    go.Scatter(
                        x=roc_df["fpr"],
                        y=roc_df["tpr"],
                        mode="lines",
                        name=f"{model_type.capitalize()}",
                        line=dict(color=colors[model_type]),
                        showlegend=(idx == 0),
                        legendgroup=model_type,
                    ),
                    row=row,
                    col=col,
                )
            else:
                print(f"Missing file in: {roc_dir}")
        # Add diagonal
        fig.add_trace(
            go.Scatter(
                x=[0, 1],
                y=[0, 1],
                mode="lines",
                line=dict(dash="dash", color="gray"),
                showlegend=False,
            ),
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text="FPR", row=row, col=col)
        fig.update_yaxes(title_text="TPR", row=row, col=col)
    fig.update_layout(title=title, height=700, width=900)
    fig.show()
    return fig

figs_roc = {}
for m in MODELS:
    figs_roc[m["prefix"]] = plot_roc_curves(
        all_results[m["prefix"]], m["results_dir"], m["prefix"],
        f"ROC Curves: Generic vs Specialised ({m['label']})"
    )

## Recall per Attack Technique

In [ ]:
technique_cols = {
    "recalltime": "Time-based",
    "recallboolean": "Boolean",
    "recallstacked": "Stacked",
    "recallerror": "Error-based",
    "recallunion": "UNION",
    "recallinsider": "Insider",
    "recallinline": "Inline",
}

def plot_recall_per_technique(results_df, title):
    if results_df.empty:
        print(f"No data available for: {title}")
        return None

    available_cols = {k: v for k, v in technique_cols.items() if k in results_df.columns}
    if not available_cols:
        print(f"WARNING: No attack technique recall columns found for: {title}")
        return None

    col_keys = list(available_cols.keys())
    techniques = list(available_cols.values())

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Generic", "Specialised"],
        horizontal_spacing=0.15,
    )
    for col_idx, model_type in enumerate(["generic", "specialised"], 1):
        subset = results_df[results_df["type"] == model_type].sort_values("dataset")
        z = [[row[c] for c in col_keys] for _, row in subset.iterrows()]
        text = [[f"{v:.1f}" for v in row] for row in z]
        fig.add_trace(
            go.Heatmap(
                z=z,
                x=techniques,
                y=subset["dataset"].tolist(),
                text=text,
                texttemplate="%{text}",
                colorscale="RdYlGn",
                zmin=0, zmax=100,
                showscale=(col_idx == 2),
                colorbar=dict(title="Recall (%)"),
            ),
            row=1, col=col_idx,
        )
    fig.update_layout(title=title, height=350, width=1000)
    fig.show()
    return fig

figs_technique = {}
for m in MODELS:
    figs_technique[m["prefix"]] = plot_recall_per_technique(
        all_results[m["prefix"]],
        f"Recall per Attack Technique: Generic vs Specialised ({m['label']})"
    )

## Recall per Statement Type

In [ ]:
stmt_type_cols = {
    "recall_select": "SELECT",
    "recall_insert": "INSERT",
    "recall_update": "UPDATE",
    "recall_delete": "DELETE",
    "recall_insider": "Insider",
}

def plot_recall_per_statement_type(results_df, title):
    if results_df.empty:
        print(f"No data available for: {title}")
        return None

    available_cols = {k: v for k, v in stmt_type_cols.items() if k in results_df.columns}
    if not available_cols:
        print(f"WARNING: No statement type recall columns found in results for: {title}")
        return None

    col_keys = list(available_cols.keys())
    stmt_types = list(available_cols.values())

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Generic", "Specialised"],
        horizontal_spacing=0.15,
    )
    for col_idx, model_type in enumerate(["generic", "specialised"], 1):
        subset = results_df[results_df["type"] == model_type].sort_values("dataset")
        z = [[row[c] for c in col_keys] for _, row in subset.iterrows()]
        text = [[f"{v:.1f}" for v in row] for row in z]
        fig.add_trace(
            go.Heatmap(
                z=z,
                x=stmt_types,
                y=subset["dataset"].tolist(),
                text=text,
                texttemplate="%{text}",
                colorscale="RdYlGn",
                zmin=0, zmax=100,
                showscale=(col_idx == 2),
                colorbar=dict(title="Recall (%)"),
            ),
            row=1, col=col_idx,
        )
    fig.update_layout(title=title, height=350, width=1000)
    fig.show()
    return fig

figs_stmt = {}
for m in MODELS:
    figs_stmt[m["prefix"]] = plot_recall_per_statement_type(
        all_results[m["prefix"]],
        f"Recall per Statement Type: Generic vs Specialised ({m['label']})"
    )

## Balanced Accuracy: Combined Comparison

In [ ]:
def plot_combined_metric(all_results, models, metric, title=None):
    """Plot a single metric comparing all approaches (generic vs specialised).

    Parameters
    ----------
    all_results : dict
        Mapping of model prefix -> results DataFrame.
    models : list of dict
        Each entry has "prefix" and "label" keys.
    metric : str
        Column name from the results DataFrames (e.g. "balanced_accuracy_per_technique",
        "fone", "rocauc", "auprc", "accuracy", "precision", "recall").
    title : str, optional
        Custom figure title. Defaults to the metric label.
    """
    all_rows = []
    for m in models:
        df = all_results[m["prefix"]]
        if not df.empty and metric in df.columns:
            tmp = df[["dataset", "type", metric]].copy()
            tmp["approach"] = m["label"]
            all_rows.append(tmp)

    if not all_rows:
        print(f"No data found for metric '{metric}'")
        return None

    combined = pd.concat(all_rows, ignore_index=True)
    label = metric_labels.get(metric, metric)
    if title is None:
        title = f"{label}: Generic vs Specialised across Feature Extractors"

    approach_order = [m["label"] for m in models]
    is_pct = combined[metric].max() > 1
    fmt = ".1f" if is_pct else ".4f"
    zmin = 0
    zmax = 100 if is_pct else 1

    fig_hm = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Generic", "Specialised"],
        horizontal_spacing=0.15,
    )
    ds_order = sorted(combined["dataset"].unique())
    for col_idx, model_type in enumerate(["generic", "specialised"], 1):
        subset = combined[combined["type"] == model_type]
        z = []
        text = []
        for ds in ds_order:
            row_z = []
            row_t = []
            for approach in approach_order:
                val = subset.loc[
                    (subset["dataset"] == ds) & (subset["approach"] == approach), metric
                ]
                v = val.values[0] if len(val) else float("nan")
                row_z.append(v)
                row_t.append(f"{v:{fmt}}")
            z.append(row_z)
            text.append(row_t)
        fig_hm.add_trace(
            go.Heatmap(
                z=z,
                x=approach_order,
                y=ds_order,
                text=text,
                texttemplate="%{text}",
                colorscale="RdYlGn",
                zmin=zmin,
                zmax=zmax,
                showscale=(col_idx == 2),
                colorbar=dict(title=label),
            ),
            row=1, col=col_idx,
        )
    fig_hm.update_layout(title=title, height=350, width=700)
    fig_hm.show()
    return fig_hm


fig_ba_combined_hm = plot_combined_metric(all_results, MODELS, "balanced_accuracy_per_technique")

## Export Figures

In [ ]:
figures_dir = Path("../output/experiments/generic-vs-specialised")
figures_dir.mkdir(parents=True, exist_ok=True)

figures = {}
for m in MODELS:
    p = m["prefix"]
    figures[f"metrics_{p}"]           = figs_metrics.get(p)
    figures[f"roc_{p}"]               = figs_roc.get(p)
    figures[f"recall_technique_{p}"]  = figs_technique.get(p)
    figures[f"recall_stmt_{p}"]       = figs_stmt.get(p)
figures["balanced_accuracy_combined_heatmap"] = fig_ba_combined_hm

for name, fig in figures.items():
    if fig is not None:
        # fig.write_image(figures_dir / f"{name}.pdf")
        fig.write_image(figures_dir / f"{name}.png", scale=2)
        print(f"Exported {name}")
    else:
        print(f"Skipped {name} (no data)")